#1. Basic Tasks

In [0]:
#1
from pyspark.sql.functions import to_date, col

customers_changed = [
    (501, 'John Doe',      'john@test.com',    '111-111-1111', 'New York',    'NY', '2022-01-01'),
    (504, 'John Ali',      'john.ali@test.com','111-444-2222', 'New York',    'NY', '2023-05-15'), 
    (502, 'Jane Smith',    'jane@test.com',    '222-222-2222', 'Toronto',     'ON', '2022-01-02'),
    (503, 'Bob Johnson',   'bob@test.com',     '333-333-3333', 'Chicago',     'IL', '2022-01-03'),
    (514, 'Alice Brown',   'alice@test.com',   '444-444-4444', 'Vancouver',   'BC', '2022-01-04'),
    (515, 'Alice D',   'alice.b@d.com', '444-333-5555', 'New York',   'BC', '2024-02-10'), 
    (517, 'Charlie Davis', 'charlie@test.com', '555-555-5555', 'Houston',     'TX', '2022-01-05'),
    (520, 'Ali Davis', 'ali@test.com', '555-555-5555', 'Houston',     'TX', '2022-01-05'),
    (511, 'David Davis', 'david@test.com', '555-555-5555', 'Houston',     'TX', '2022-01-05'),
    (512, 'James Davis', 'james@test.com', '555-555-5555', 'Houston',     'TX', '2022-01-05')

]

columns = ['customer_id', 'name', 'email', 'phone', 'city', 'state', 'signup_date']

df_customers_changed = spark.createDataFrame(customers_changed, columns)

df_customers_changed = df_customers_changed.withColumn('signup_date', to_date(col('signup_date')))

df_customers_changed.write.mode('overwrite').saveAsTable('cyntexa_dev.sales.silver_customers_changed')

In [0]:
%sql
merge into cyntexa_dev.sales.silver_customers target
using 
(select * from cyntexa_dev.sales.silver_customers_changed
qualify rank() over(partition by customer_id order by signup_date desc) = 1)
source
on target.customer_id = source.customer_id
when matched then 
update set target.customer_id = source.customer_id, target.name = source.name, target.email = source.email, target.phone = source.phone, target.city = source.city, target.state = source.state, target.signup_date = source.signup_date

when not matched then 
insert (customer_id, name, email, phone, city, state, signup_date) values (source.customer_id, source.name, source.email, source.phone, source.city, source.state, source.signup_date)

In [0]:
%sql
select * from cyntexa_dev.sales.silver_customers where customer_id = 1

In [0]:
%sql
insert into table cyntexa_dev.sales.silver_customers(customer_id, name, email, phone, city, state, signup_date) 
values(1, 'John Doe', 'john@test.com', '222-111-1111', 'New York', 'NY', '2025-01-01')

In [0]:
%sql
select * from cyntexa_dev.sales.silver_customers where customer_id = 1

In [0]:
%sql
--2
grant select on table cyntexa_dev.sales.silver_customers to `Data Engineers`

In [0]:
%sql
create or replace view cyntexa_dev.sales.silver_customers_masked as
select
  customer_id, name, city, state,
  concat(substring(email, 1, 2), '***@***.com') as email
from cyntexa_dev.sales.silver_customers

In [0]:
%sql
grant select on view cyntexa_dev.sales.silver_customers_masked to `Data Analyst`

In [0]:
%sql
--3
select * from system.billing.usage
order by usage_date desc
limit 5

A DBU is a normalized unit of processing power it is not billing for a specific machine or hours rented, it is billing for how much compute work Databricks software did based on the size of your cluster and how long it ran. It is metered per second and summed across every node in a cluster so a bigger cluster or a longer-running job burns more DBUs

#2. Intermediate Tasks

In [0]:
%sql
--4
create or replace table cyntexa_dev.sales.customers_silver_scd2(
   customer_id int,
   name string,
   email string,
   phone string,
   city string,
   state string,
   signup_date date,
   is_current boolean,
   end_date date
)

In [0]:
%sql
merge into cyntexa_dev.sales.customers_silver_scd2 t
using cyntexa_dev.sales.customers_changed s
on t.customer_id = s.customer_id
when matched and (
    t.name != s.name or
    t.email != s.email or
    t.phone != s.phone or
    t.city != s.city or
    t.state != s.state or
    t.signup_date != s.signup_date
)then
update set
  t.is_current = false,
  t.end_date = current_date()

In [0]:
%sql
merge into cyntexa_dev.sales.customers_silver_scd2 t
using (
  select s.*
  from cyntexa_dev.sales.customers_changed s
  left join cyntexa_dev.sales.customers_silver_scd2 t
    on s.customer_id = t.customer_id  and t.is_current = true
  where t.customer_id  is null
) s
on t.customer_id  = s.customer_id  and t.is_current = true
when not matched then
insert (customer_id, name, email, phone, city, state, signup_date, is_current, end_date)
values (s.customer_id, s.name, s.email, s.phone, s.city, s.state, s.signup_date, true, null)

In [0]:
%sql
--5
select
  customer_id,
  name,
  city,
  state,
  signup_date,
  end_date,
  is_current
from cyntexa_dev.sales.customers_silver_scd2
where customer_id = 4
  and '2026-03-01' >= signup_date
  and ('2026-03-01' < end_date or end_date is null)

6
- All-purpose compute is for interactive notebook work and stays running, cost more per DBU. 
- Job compute spins up only when a scheduled job runs then shuts down immediately making it much cheaper. Cyntexa's nightly pipeline is automated and unattended it should use job compute same code, same result, but lower cost and no risk of an idle cluster being left running by mistake overnight.

#3. Advanced Tasks

#### 7- Data Governance Model for Cyntexa

1. Unity Catalog Namespace

Cyntexa follows the three-level Unity Catalog namespace:
cyntexa_dev.sales.customers_silver_scd2

2. Sensitive / PII Columns

For the customers_silver_scd2 table:
Column - email,phone  
Protection - Mask for analysts  

3. Unity Catalog Groups
Data Engineer Group - Data Engineers
* Can access the customer table.
* Can see original email,phone values.

Data Analyst Group - Data Analysts
* Can query the customer table.
* Cannot see raw PII.
* Email and phone are masked.

4. PII Protection

Column masking is applied to email and phone.

For a larger production environment, Cyntexa can use Unity Catalog ABAC with governed PII tags and centralized masking policies.

5. Access Auditing
Access is audited at two levels.
Permission audit
sql
SHOW GRANTS ON TABLE cyntexa_dev.customer.customers_silver_scd2

In [0]:
%sql
--8
merge into cyntexa_dev.sales.customers_silver_scd2 t
using cyntexa_dev.sales.customers_changed s
on t.customer_id = s.customer_id
when matched and (
    t.name <=> s.name or
    t.email <=> s.email or
    t.phone <=> s.phone or
    t.city <=> s.city or
    t.state <=> s.state or
    t.signup_date <=> s.signup_date
)then
update set
  t.is_current = false,
  t.end_date = current_date()

In [0]:
%sql
merge into cyntexa_dev.sales.customers_silver_scd2 t
using (
  select s.*
  from cyntexa_dev.sales.customers_changed s
  left join cyntexa_dev.sales.customers_silver_scd2 t
    on s.customer_id = t.customer_id  and t.is_current = true
  where t.customer_id  is null
  or not (
       t.name        <=> s.name and
       t.email       <=> s.email and
       t.phone       <=> s.phone and
       t.city        <=> s.city and
       t.state       <=> s.state and
       t.signup_date <=> s.signup_date
     )
) s
on t.customer_id  = s.customer_id  and t.is_current = true
when not matched then
insert (customer_id, name, email, phone, city, state, signup_date, is_current, end_date)
values (s.customer_id, s.name, s.email, s.phone, s.city, s.state, s.signup_date, true, null)

In [0]:
%sql
select customer_id, count(*) as current_version_count
from cyntexa_dev.sales.customers_silver_scd2
where is_current = true
group by customer_id
having count(*) > 1

In [0]:
%sql
--9
with months as (
  select explode(sequence(to_date('2026-01-01'), to_date('2026-09-01'), interval 1 month)) as snapshot_date
)

select
  m.snapshot_date,
  count(distinct t.customer_id) as active_customers
from months m
join cyntexa_dev.sales.customers_silver_scd2 t
  on m.snapshot_date >= t.signup_date
  and (m.snapshot_date < t.end_date or t.end_date is null)
group by m.snapshot_date
order by m.snapshot_date

In [0]:
%sql
select
  snapshot_date,
  active_customers,
  active_customers - lag(active_customers) over (order by snapshot_date) as change
from (
  select
    m.snapshot_date,
    count(distinct t.customer_id) as active_customers
  from (select explode(sequence(to_date('2026-01-01'), to_date('2026-09-01'), interval 1 month)) as snapshot_date) m
  join cyntexa_dev.sales.customers_silver_scd2 t
    on m.snapshot_date >= t.signup_date
    and (m.snapshot_date < t.end_date or t.end_date is null)
  group by m.snapshot_date
)
order by snapshot_date;